In [1]:
import numpy as np
import bricks2marble as b2m

## Example Data

We simulate a model that returns a sequence of Tiberius-HMM states.
We set the total genome length to 3000 and divide this into three sequences of length 1000. A random fasta sequence of the same length will simulate the input to our model.

In [2]:
fasta = b2m.struct.FASTA([
    b2m.struct.Sequence(
        np.random.randint(9, size=(1000,)),
        name=f"seq_{i}",
        start=i*1000,
        end=(i+1)*1000,
    )
    for i in range(3)
])
fasta.resample(1000)

FASTA('seq_0'[0:1000], 'seq_1'[1000:2000], 'seq_2'[2000:3000])

In [3]:
example_fwd = np.array([]
    # IR - exon - I0 - exon - I2 - exon - IR
    + 100*[0] + [7] + 20*[5, 6, 4] + [8] + 150*[1] + [11] + 10*[4, 5, 6]
    + [10] + 140*[3] + [13] + 30*[6, 4, 5] + [14] + 424*[0]

    # IR - exon - I0 - exon - I1 - exon - IR
    + 100*[0] + [7] + 15*[5, 6, 4] + [8] + 50*[1] + [11] + 40*[4, 5, 6]
    + [4, 5, 9] + 200*[2] + [12] + 35*[5, 6, 4] + [5, 14] + 500*[0]

    # IR - exon - I1 - exon - IR
    + 100*[0] + [7] + 15*[5, 6, 4] + [9] + 200*[2]
    + 35*[5, 6, 4] + [5, 14] + 417*[0]
)
example_bwd = np.array([]
    # IR - exon - I1 - exon - IR
    + 100*[0] + [7] + 15*[5, 6, 4] + [9] + 200*[2]
    + 35*[5, 6, 4] + [5, 14] + 417*[0]

    # IR - exon - I0 - exon - I1 - exon - IR
    + 100*[0] + [7] + 15*[5, 6, 4] + [8] + 50*[1] + [11] + 20*[4, 5, 6]
    + [4, 5, 9] + 200*[2] + [12] + 55*[5, 6, 4] + [5, 14] + 500*[0]

    # IR - exon - I1 - exon - IR
    + 100*[0] + [7] + 58*[5, 6, 4] + [9] + 200*[2]
    + 35*[5, 6, 4] + [5, 14] + 417*[0]
)
example_fwd.shape, example_bwd.shape

((3000,), (3000,))

## Predict function
A pseudo predict function that would normally be a model call.

In [4]:
def predict_func(fasta: b2m.struct.FASTA) -> tuple[np.ndarray, np.ndarray]:
    returned_fwd = []
    returned_bwd = []
    for seq in fasta:
        for batch in seq.nuc:
            returned_fwd.append(example_fwd[seq.start:seq.end])
            returned_bwd.append(example_bwd[seq.start:seq.end])
    return np.array(returned_fwd), np.array(returned_bwd)

## Annotation of a fasta sequence
Using the above defined predict_func gives us a GTF annotation, which we can check by creating the
corresponding list of GTFEntry objects or writing it directly to a .gtf file.

In [5]:
annotation = b2m.tools.GTF_from_model(
    fasta=fasta,
    predict_func=predict_func,
    model_name="Exampler",
)

[0.0000s] Start initial prediction of 3 sequences.
[0.0004s] Searching for errors.
[0.0009s] Forming regions.
[0.0018s] Creating GTF entries.
[0.0025s] Finalizing.
[0.0029s] Done.


In [6]:
entries = annotation.to_list()
# annotation.to_gtf("output.gtf")
entries

[GTFEntry(name='seq_0', start=101, end=576, strand='+', source='Exampler', feature=<FeatureType.Gene: 'gene'>, score=None, frame=None, attributes='gene_id "g1";'),
 GTFEntry(name='seq_0', start=101, end=576, strand='+', source='Exampler', feature=<FeatureType.Transcript: 'transcript'>, score=None, frame=None, attributes='gene_id "g1"; transcript_id "g1.t1";'),
 GTFEntry(name='seq_0', start=101, end=103, strand='+', source='Exampler', feature=<FeatureType.StartCodon: 'start_codon'>, score=None, frame=0, attributes='gene_id "g1"; transcript_id "g1.t1";'),
 GTFEntry(name='seq_0', start=101, end=162, strand='+', source='Exampler', feature=<FeatureType.CDS: 'CDS'>, score=None, frame=0, attributes='gene_id "g1"; transcript_id "g1.t1"; cds_type "initial";'),
 GTFEntry(name='seq_0', start=163, end=312, strand='+', source='Exampler', feature=<FeatureType.Intron: 'intron'>, score=None, frame=1, attributes='gene_id "g1"; transcript_id "g1.t1";'),
 GTFEntry(name='seq_0', start=313, end=344, strand

# Repredictions

Now a case where predictions did not match at some sequence boundaries.

In [7]:
fasta = b2m.struct.FASTA([
    b2m.struct.Sequence(
        np.random.randint(9, size=(70,)),
        name=f"chr1",
    )
])
fasta.resample(10)

FASTA('chr1'[0:70])

In [8]:
predict_fwd = np.array([
    0,  0,  0,  0,  0,  0,  7,  5,  9,  2,
    2,  2, 12,  5,  6,  4,  5, 14,  0,  0,  # fail
    6,  4,  5,  6,  4,  5,  6, 10,  3,  3,  # fail
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  # fail
    2,  2,  2,  2,  2,  12, 5,  14, 0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
])
predict_bwd = np.array([
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
])
predict_fwd.shape, predict_bwd.shape

((70,), (70,))

In [9]:
repredict_fwd = np.array([
    4,  5, 14,  0,  0,  0,  7,  5,  6,  4,
    # 5,  6, 10,  3,  3,  3, 13,  6,  4,  5,
    5, 14,  0,  0,  0,  0,  0,  0,  0,  0,  # alternative to previous row
    0,  0,  7,  5,  6,  4,  5,  9,  2,  2,
])
repredict_bwd = np.array([
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
    0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
])
repredict_fwd.shape, repredict_bwd.shape

((30,), (30,))

In [10]:
def predict_func(fasta: b2m.struct.FASTA) -> tuple[np.ndarray, np.ndarray]:
    return predict_fwd.reshape(7, 10), predict_bwd.reshape(7, 10)

def repredict_func(fasta: b2m.struct.FASTA) -> tuple[np.ndarray, np.ndarray]:
    return repredict_fwd.reshape(3, 10), repredict_bwd.reshape(3, 10)


In [11]:
annotation = b2m.tools.GTF_from_model(
    fasta=fasta,
    predict_func=predict_func,
    repredict_func=repredict_func,
    model_name="Exampler",
    liberal=True,
)

[0.0000s] Start initial prediction of 7 sequences.
[0.0004s] Searching for errors.
[0.0013s] Mismatches: 3 (+) | 0 (-) | 0 (+/-). Repredicting 3 sequences.
[0.0015s] Merging repredictions.
[0.0019s] Forming regions.
[0.0024s] Creating GTF entries.
[0.0028s] Finalizing.
[0.0033s] Done.


In [12]:
entries = annotation.to_list()
# annotation.to_gtf("output.gtf")
entries

[GTFEntry(name='chr1', start=7, end=18, strand='+', source='Exampler', feature=<FeatureType.Gene: 'gene'>, score=None, frame=None, attributes='gene_id "g1";'),
 GTFEntry(name='chr1', start=7, end=18, strand='+', source='Exampler', feature=<FeatureType.Transcript: 'transcript'>, score=None, frame=None, attributes='gene_id "g1"; transcript_id "g1.t1";'),
 GTFEntry(name='chr1', start=7, end=9, strand='+', source='Exampler', feature=<FeatureType.CDS: 'CDS'>, score=None, frame=0, attributes='gene_id "g1"; transcript_id "g1.t1"; cds_type "initial";'),
 GTFEntry(name='chr1', start=7, end=9, strand='+', source='Exampler', feature=<FeatureType.StartCodon: 'start_codon'>, score=None, frame=0, attributes='gene_id "g1"; transcript_id "g1.t1";'),
 GTFEntry(name='chr1', start=10, end=12, strand='+', source='Exampler', feature=<FeatureType.Intron: 'intron'>, score=None, frame=0, attributes='gene_id "g1"; transcript_id "g1.t1";'),
 GTFEntry(name='chr1', start=13, end=18, strand='+', source='Exampler',